# SQL with Spark

In [ ]:
# import packages
import pyspark
from pyspark.sql import SparkSession

In [6]:
# Initialize SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("pyspark_sql") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/19 15:06:01 WARN Utils: Your hostname, JACK2000-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.158 instead (on interface en0)
26/06/19 15:06:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/19 15:06:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1) Loading the data

In [8]:
df_green = spark.read.parquet('data/pq/green/*/*')

df_yellow = spark.read.parquet('data/pq/yellow/*/*')

26/06/19 15:10:08 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

## 2) Finding common columns

In [9]:
## 2.1) Rename columns to have a common schema  
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [10]:
# 2.2) Find common columns
common_colums = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)

## 3) Combining the data (Yellow + Green)

In [11]:
# 3.1) Add new column to identify the service type (green or yellow) to tell the diference
from pyspark.sql import functions as F

df_green_sel = df_green.select(common_colums).withColumn('service_type', F.lit('green'))

df_yellow_sel = df_yellow.select(common_colums).withColumn('service_type', F.lit('yellow'))

In [12]:
# 3.2) Combine the data using unionAll
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [13]:
# 3.3) Verify the merge
df_trips_data.groupBy('service_type').count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 1734051|
|      yellow|24648499|
+------------+--------+



## 4) Query with SQL

In [ ]:
# 4.1) Tell Spark that this DataFrame is a table and we can use SQL to query it
df_trips_data.createOrReplaceTempView('trips_data')

/Users/jack2000/data-engineer-projects/data-engineering-zoomcamp-2026/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [15]:
# 4.2) Now we can treat it as a table and query it with SQL
spark.sql("""
select
    service_type,
    count(1) as trips
from trips_data
group by service_type
"""
).show()

+------------+--------+
|service_type|   trips|
+------------+--------+
|       green| 1734051|
|      yellow|24648499|
+------------+--------+



In [16]:
# 4.3) Execute query from Module 4
df_result = spark.sql("""
SELECT 
    -- Reveneue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_montly_passenger_count,
    AVG(trip_distance) AS avg_montly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

# Show the result
df_result \
    .select('revenue_zone', 'revenue_month', 'service_type', 'revenue_monthly_total_amount', 'avg_montly_passenger_count') \
    .show()

+------------+-------------------+------------+----------------------------+--------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_total_amount|avg_montly_passenger_count|
+------------+-------------------+------------+----------------------------+--------------------------+
|         221|2020-01-01 00:00:00|       green|                      474.28|        1.2857142857142858|
|          35|2020-01-01 00:00:00|       green|           56871.41999999995|        1.1042183622828785|
|          29|2020-01-01 00:00:00|       green|          27554.409999999934|        1.1295681063122924|
|         265|2020-01-01 00:00:00|       green|          14043.179999999993|        1.4027149321266967|
|         156|2020-01-01 00:00:00|       green|                     4634.34|        1.4090909090909092|
|         245|2020-01-01 00:00:00|       green|                       83.05|                       6.0|
|         244|2020-01-01 00:00:00|       green|          232629.

## 5) Save the result

In [17]:
# Using coalesce(1) to write the result into a single parquet file
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')